# Prototype 22 — Three-way System-1 benchmark: Laya vs. PrismNLI vs. OpenDecision

**Question:** Which self-hostable System-1 decision engine (if any) should
Vastra ship in Phase 1 instead of paying for Jev via OpenRouter?

**Candidates:**

| Name | Model / Package | License | Loading | Native primitives? |
|---|---|---|---|---|
| **Laya** | `convaiinnovations/laya` (ModernBERT-large, 421M) — `pip install laya` | Apache 2.0 | `agent = laya.load(...)` | Yes — `choice/noul/score` |
| **PrismNLI** | `MoritzLaurer/deberta-v3-large-zeroshot-v2.0` (DeBERTa-v3-large, 400M) — `pip install transformers` | MIT | `pipeline("zero-shot-classification", ...)` | No — we adapt NLI → primitives |
| **OpenDecision** | `deepanwadhwa/OpenDecision` (wraps `MoritzLaurer/ModernBERT-large-zeroshot-v2.0`, 400M) — `pip install OpenDecision` | Apache 2.0 | `OpenDecisionEngine()` | Yes — `choice/noul/score/relation` |

**Baseline:** cached Jev outputs from nb20 (`prototypes/cache/20_jev_*.json`)
on the same 10 fixtures × ~32 outfits. We never re-run Jev in this notebook.

**Decision criterion — all three gates must pass, per candidate:**
1. **Choice agreement with Jev ≥ 90%** on top-1 pick per fixture.
2. **Median CPU latency < 1s per decision.**
3. **Confidence median within ±0.05 of Jev's**, and no bimodal collapse
   to 0/1.

**Winner selection across candidates that pass all gates:**
- Highest `choice_agreement × (1 − ECE)` on the noul primitive.
- Tie-break on deployment simplicity (pip-installable > local server;
  fewer Python-version constraints > more).

**If none pass** → recommend keeping Jev.

**Sources:**
- Laya card: https://huggingface.co/convaiinnovations/laya
- PrismNLI card: https://huggingface.co/MoritzLaurer/deberta-v3-large-zeroshot-v2.0
- OpenDecision repo: https://github.com/deepanwadhwa/OpenDecision
- OpenDecision default model: https://huggingface.co/MoritzLaurer/ModernBERT-large-zeroshot-v2.0
- Prior work: `prototypes/notebooks/20_jev_suggest_rank.ipynb`
- Plan: `notes/ai-generated/jev-integration-plan.md`
- Epic: #276

## Install (one-time, before running)

```
pip install laya                       # Laya SDK
pip install transformers               # PrismNLI (also brings torch on most stacks)
pip install OpenDecision               # OpenDecision — REQUIRES PYTHON 3.13+
pip install huggingface_hub sentencepiece
```

Notes:
- Model weights are ~800 MB (Laya) + ~880 MB (PrismNLI DeBERTa-v3-large)
  + ~800 MB (OpenDecision ModernBERT) — plan for **~2.5 GB total** disk
  in `~/.cache/huggingface/` plus mirrored copies under
  `prototypes/cache/<candidate>-model/`.
- If `laya.load()` hangs, run with `USE_TF=0` (documented on the Laya card).
- **OpenDecision requires Python 3.13+** per its pyproject; if your kernel
  is 3.11/3.12 that cell will raise a `SystemExit` with a clear message.


## 1. Setup — imports, env, paths


In [ ]:
from __future__ import annotations

import json
import os

# Guard against the TF/abseil deadlock documented on the Laya model card
# (transformers auto-probes TF at import; abseil can deadlock on some setups).
os.environ.setdefault("USE_TF", "0")
os.environ.setdefault("TRANSFORMERS_NO_ADVISORY_WARNINGS", "1")

import sqlite3
import statistics
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

REPO_ROOT = Path.cwd().parent.parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Load .env if present (matches nb20/nb21 convention)
env_path = REPO_ROOT / ".env"
if env_path.exists():
    for line in env_path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))

CLOSET_DATA_DIR = Path(os.environ.get("CLOSET_DATA_DIR", str(Path.home() / ".closet")))
# nb21 pattern: fall back to repo-local .closet/closet.db if home dir is empty.
CANDIDATE_DBS = [
    CLOSET_DATA_DIR / "closet.db",
    REPO_ROOT / ".closet" / "closet.db",
]
CLOSET_DB = next((p for p in CANDIDATE_DBS if p.exists()), CANDIDATE_DBS[0])

OUT_DIR = REPO_ROOT / "prototypes" / "outputs" / "22"
OUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR = REPO_ROOT / "prototypes" / "cache"
CACHE_DIR.mkdir(exist_ok=True)
LAYA_MODEL_CACHE = CACHE_DIR / "laya-model"
PRISMNLI_MODEL_CACHE = CACHE_DIR / "prismnli-model"
OPENDECISION_MODEL_CACHE = CACHE_DIR / "opendecision-model"
for p in (LAYA_MODEL_CACHE, PRISMNLI_MODEL_CACHE, OPENDECISION_MODEL_CACHE):
    p.mkdir(exist_ok=True)

# Which candidates to run. Flip a value to False to skip (e.g. skipping
# OpenDecision on a Python 3.11 kernel). At least one must be True.
RUN_LAYA = os.environ.get("RUN_LAYA", "1") == "1"
RUN_PRISMNLI = os.environ.get("RUN_PRISMNLI", "1") == "1"
RUN_OPENDECISION = os.environ.get("RUN_OPENDECISION", "1") == "1"

print(f"CLOSET_DB:               {CLOSET_DB}  exists={CLOSET_DB.exists()}")
print(f"OUT_DIR:                 {OUT_DIR}")
print(f"Laya cache:              {LAYA_MODEL_CACHE}          run={RUN_LAYA}")
print(f"PrismNLI cache:          {PRISMNLI_MODEL_CACHE}      run={RUN_PRISMNLI}")
print(f"OpenDecision cache:      {OPENDECISION_MODEL_CACHE}  run={RUN_OPENDECISION}")
print(f"Python:                  {sys.version.split()[0]}")


## 2a. Laya SDK load — one-time (~808 MB download + 7-10s CPU build)

Per the Laya model card, the recommended production path is:

```python
import laya
agent = laya.load("convaiinnovations/laya")
result = agent.predict(state, questions)
```

- First call downloads and builds the ModernBERT-large checkpoint into
  the HuggingFace cache (~808 MB, English base). Subsequent calls are
  fast — the model stays resident in this Python process.
- On CPU (preloaded), per-request latency is documented as **193-464 ms**;
  on T4 GPU it's ~40 ms.
- No auth needed — Apache 2.0, public HF weights.

**Checkpoint toggle:** `LAYA_CHECKPOINT` env var (defaults to base).
Set to `"convaiinnovations/laya-typed-decisions"` if you want to
A/B against the specialist — but expect worse numbers on
out-of-distribution wardrobe decisions per its own model card.

In [ ]:
laya_agent = None
LAYA_CHECKPOINT = os.environ.get("LAYA_CHECKPOINT", "convaiinnovations/laya")

if not RUN_LAYA:
    print("SKIP Laya (RUN_LAYA=0)")
else:
    print(f"Loading Laya checkpoint: {LAYA_CHECKPOINT}")
    print("(First run: ~808 MB download + 7-10s ONNX/SafeTensors rebuild)")

    try:
        import laya  # noqa: F401
    except ImportError as e:
        raise SystemExit(
            "Laya SDK not installed. Run `pip install laya` first, then re-run this cell.\n"
            f"Import error: {e}"
        )

    t0 = time.perf_counter()
    try:
        laya_agent = laya.load(LAYA_CHECKPOINT)
    except Exception as e:
        # TODO: verify — model card mentions `laya.Router` as an alternative path.
        # If `.load()` fails, try:
        #   from laya import Router
        #   router = Router(preload=True); router.predict(state, questions)
        raise SystemExit(
            f"laya.load('{LAYA_CHECKPOINT}') failed: {type(e).__name__}: {e}\n"
            "If this is a TF/abseil deadlock, restart the kernel with USE_TF=0.\n"
            "If it's a network error, check `~/.cache/huggingface/` disk space."
        )
    laya_load_s = time.perf_counter() - t0
    print(f"Loaded in {laya_load_s:.2f}s")

    # Warm the model with a trivial predict so the first real fixture doesn't
    # eat JIT/tokenizer warm-up cost.
    t0 = time.perf_counter()
    _ = laya_agent.predict({"note": "warm-up"}, {"warm": {"type": "noul", "instructions": "Is the sky blue?"}})
    print(f"Warm-up predict: {(time.perf_counter() - t0) * 1000:.1f} ms")


## 2b. PrismNLI (DeBERTa zero-shot) load — one-time (~880 MB download)

`MoritzLaurer/deberta-v3-large-zeroshot-v2.0` — 400M params, MIT license,
mean F1 macro **0.676** across 28 datasets (vs 0.497 for bart-large-mnli).
Loaded via the standard `transformers` zero-shot-classification pipeline.

We snapshot-download weights into `prototypes/cache/prismnli-model/` so
the notebook is portable across HuggingFace cache locations.

**NLI → primitive translation strategy:**
- **Choice:** single pipeline call with `candidate_labels = [outfit_id, ...]`,
  hypothesis template `"This outfit is the best pick for the given context: {}"`,
  `multi_label=False`. Pipeline returns per-label scores summing to 1
  (softmax over entailment logits), which map 1:1 to Jev's `probabilities`.
- **Noul:** N per-outfit binary classifications with
  `candidate_labels=["yes", "no"]` and hypothesis `"This outfit is a good
  pick for {context}: yes"`. Take the `yes` probability as the noul score.
  Batched via a single pipeline call over the outfit list where possible.

**Chunking:** the pipeline has no hard label limit but attention over
long `candidate_labels` lists slows super-linearly. We chunk at 20 like
Laya, then tie-break the winners.

In [ ]:
prismnli_pipe = None
PRISMNLI_CHECKPOINT = os.environ.get(
    "PRISMNLI_CHECKPOINT", "MoritzLaurer/deberta-v3-large-zeroshot-v2.0"
)

if not RUN_PRISMNLI:
    print("SKIP PrismNLI (RUN_PRISMNLI=0)")
else:
    print(f"Loading PrismNLI checkpoint: {PRISMNLI_CHECKPOINT}")
    print("(First run: ~880 MB download of DeBERTa-v3-large weights)")

    try:
        from huggingface_hub import snapshot_download
        from transformers import pipeline
    except ImportError as e:
        raise SystemExit(
            "transformers / huggingface_hub not installed. Run:\n"
            "  pip install transformers huggingface_hub sentencepiece\n"
            f"Import error: {e}"
        )

    t0 = time.perf_counter()
    try:
        model_path = snapshot_download(
            repo_id=PRISMNLI_CHECKPOINT,
            cache_dir=str(PRISMNLI_MODEL_CACHE),
        )
        prismnli_pipe = pipeline(
            "zero-shot-classification",
            model=model_path,
            device=-1,  # CPU — matches Laya's default so latency is comparable
        )
    except Exception as e:
        raise SystemExit(
            f"PrismNLI load failed: {type(e).__name__}: {e}\n"
            "Check disk space in prototypes/cache/prismnli-model/ (~880 MB needed)."
        )
    prismnli_load_s = time.perf_counter() - t0
    print(f"Loaded in {prismnli_load_s:.2f}s")

    # Warm-up
    t0 = time.perf_counter()
    _ = prismnli_pipe(
        "The sky is blue.",
        candidate_labels=["true", "false"],
        hypothesis_template="This statement is {}.",
        multi_label=False,
    )
    print(f"Warm-up classify: {(time.perf_counter() - t0) * 1000:.1f} ms")


## 2c. OpenDecision load — one-time (~800 MB ModernBERT download)

`deepanwadhwa/OpenDecision` (pypi: `OpenDecision`, Apache 2.0) is the
only self-hostable candidate besides Laya that ships **native**
`choice`/`noul`/`score`/`relation` primitives — same API shape as Jev.
Backbone is `MoritzLaurer/ModernBERT-large-zeroshot-v2.0` (400M params,
Apache 2.0), 2-3× faster than DeBERTa-v3 per the ModernBERT card.

**API used here** (verified against `src/opendecision/engine.py`):
```python
engine.choice(state=..., instructions=..., criteria={id: desc, ...})
    -> {"type": "choice", "choice": id, "probabilities": {id: p}, "confidence": float}

engine.noul(state=..., instructions=..., criteria=None)
    -> {"type": "noul", "noul": float}   # entailment probability 0..1
```

**Confidence semantics:** OpenDecision derives `confidence` from
normalised entropy over `probabilities` (1.0 = concentrated, 0.0 = uniform).
Jev's `confidence` field is not documented as entropy-based but behaves
numerically similarly in nb20 outputs — we compare medians directly and
accept ±0.05 band per the gate spec.

**Python requirement:** OpenDecision's pyproject pins `python>=3.13`.
If your kernel is 3.11/3.12 this cell raises a clear `SystemExit` so
Aryan knows to skip via `RUN_OPENDECISION=0` or upgrade the kernel.

In [ ]:
opendecision_engine = None

if not RUN_OPENDECISION:
    print("SKIP OpenDecision (RUN_OPENDECISION=0)")
elif sys.version_info < (3, 13):
    print(
        f"SKIP OpenDecision — needs Python 3.13+, kernel is {sys.version.split()[0]}.\n"
        "Set RUN_OPENDECISION=0 to silence this warning, or reinstall kernel."
    )
    RUN_OPENDECISION = False
else:
    print("Loading OpenDecision engine (ModernBERT-large-zeroshot-v2.0 backbone)")
    print("(First run: ~800 MB weight download)")

    try:
        # TODO: verify — engine.__init__ auto-downloads from HF on first call.
        # If we want deterministic caching under prototypes/cache/, we may need
        # to pre-snapshot_download the backbone and point HF_HOME to it.
        os.environ.setdefault("HF_HOME", str(OPENDECISION_MODEL_CACHE))
        from opendecision import OpenDecisionEngine
    except ImportError as e:
        raise SystemExit(
            "OpenDecision not installed. Run:\n"
            "  pip install OpenDecision\n"
            "(Requires Python 3.13+.)\n"
            f"Import error: {e}"
        )

    t0 = time.perf_counter()
    try:
        opendecision_engine = OpenDecisionEngine()
    except Exception as e:
        raise SystemExit(
            f"OpenDecisionEngine() failed: {type(e).__name__}: {e}\n"
            "Check disk space in prototypes/cache/opendecision-model/ (~800 MB)."
        )
    od_load_s = time.perf_counter() - t0
    print(f"Loaded in {od_load_s:.2f}s")

    # Warm-up — trivial choice call to force lazy weight loading
    t0 = time.perf_counter()
    _ = opendecision_engine.choice(
        state="warm-up",
        instructions="Pick one.",
        criteria={"a": "the letter a", "b": "the letter b"},
    )
    print(f"Warm-up choice: {(time.perf_counter() - t0) * 1000:.1f} ms")


## 3. Data loaders — closet items + outfits (copied verbatim from nb20)


In [ ]:
def pick_active_user(db_path: Path) -> str | None:
    if not db_path.exists():
        return None
    conn = sqlite3.connect(f"file:{db_path}?mode=ro", uri=True)
    row = conn.execute(
        "SELECT user_id, COUNT(*) c FROM items GROUP BY user_id ORDER BY c DESC LIMIT 1"
    ).fetchone()
    conn.close()
    return row[0] if row else None


def fetch_closet_items(db_path: Path, user_id: str) -> list[dict]:
    conn = sqlite3.connect(f"file:{db_path}?mode=ro", uri=True)
    conn.row_factory = sqlite3.Row
    rows = conn.execute(
        """
        SELECT id, category, subcategory, colors, brand, material, season,
               occasion, tags, ai_description, image_uri, wear_context
        FROM items
        WHERE user_id = ?
        ORDER BY id
        """,
        (user_id,),
    ).fetchall()
    conn.close()
    out = []
    for r in rows:
        d = dict(r)
        for jf in ("colors", "season", "occasion", "tags"):
            try:
                d[jf] = json.loads(d[jf]) if d[jf] else []
            except (json.JSONDecodeError, TypeError):
                d[jf] = []
        out.append(d)
    return out


def fetch_closet_outfits(db_path: Path, user_id: str) -> list[dict]:
    conn = sqlite3.connect(f"file:{db_path}?mode=ro", uri=True)
    conn.row_factory = sqlite3.Row
    rows = conn.execute(
        """
        SELECT id, name, item_ids, occasion, season, notes
        FROM outfits
        WHERE user_id = ?
        ORDER BY id
        """,
        (user_id,),
    ).fetchall()
    conn.close()
    out = []
    for r in rows:
        d = dict(r)
        for jf in ("item_ids", "season"):
            try:
                d[jf] = json.loads(d[jf]) if d[jf] else []
            except (json.JSONDecodeError, TypeError):
                d[jf] = []
        out.append(d)
    return out


def outfit_criteria(outfits: list[dict], by_id: dict[str, dict]) -> dict[str, str]:
    """Choice.criteria: {outfit_id: short description of what's in it}.
    Copied verbatim from nb20 §7 so all candidates see the same criteria
    text Jev did."""
    out = {}
    for o in outfits:
        item_descs = []
        for iid in o.get("item_ids", []):
            it = by_id.get(iid)
            if not it:
                continue
            desc = it.get("category", "?")
            if it.get("subcategory"):
                desc += f":{it['subcategory']}"
            if it.get("colors"):
                desc += f" ({'/'.join(it['colors'][:2])})"
            item_descs.append(desc)
        name = o.get("name") or "Unnamed"
        out[o["id"]] = f"{name} — {', '.join(item_descs) or '(empty)'}"
    return out


ACTIVE_USER = pick_active_user(CLOSET_DB)
print(f"Active user (most items): {ACTIVE_USER}")

items = fetch_closet_items(CLOSET_DB, ACTIVE_USER) if ACTIVE_USER else []
outfits = fetch_closet_outfits(CLOSET_DB, ACTIVE_USER) if ACTIVE_USER else []
items_by_id = {it["id"]: it for it in items}

print(f"Items:   {len(items)}")
print(f"Outfits: {len(outfits)}")


## 4. Fixtures — copied verbatim from nb20


In [ ]:
FIXTURES = [
    {
        "id": "rainy_work_morning",
        "weather": {"temperature": 13, "condition": "Rainy", "precipitation": 5.2, "windSpeed": 15},
        "occasion": "office",
        "theme": None,
        "timeOfDay": "morning",
        "dayOfWeek": "Tuesday",
    },
    {
        "id": "hot_brunch_weekend",
        "weather": {"temperature": 32, "condition": "Sunny", "precipitation": 0, "windSpeed": 5},
        "occasion": "brunch",
        "theme": "weekend, want to feel breezy",
        "timeOfDay": "midday",
        "dayOfWeek": "Saturday",
    },
    {
        "id": "cold_formal_interview",
        "weather": {"temperature": -2, "condition": "Snow", "precipitation": 1, "windSpeed": 20},
        "occasion": "job interview",
        "theme": None,
        "timeOfDay": "morning",
        "dayOfWeek": "Monday",
    },
    {
        "id": "warm_casual_errands",
        "weather": {"temperature": 22, "condition": "Cloudy", "precipitation": 0, "windSpeed": 8},
        "occasion": "running errands",
        "theme": None,
        "timeOfDay": "afternoon",
        "dayOfWeek": "Saturday",
    },
    {
        "id": "mild_evening_date",
        "weather": {"temperature": 17, "condition": "Clear", "precipitation": 0, "windSpeed": 3},
        "occasion": "dinner date",
        "theme": "want to look put-together but not overdressed",
        "timeOfDay": "evening",
        "dayOfWeek": "Friday",
    },
    {
        "id": "cold_wfh",
        "weather": {"temperature": 4, "condition": "Cloudy", "precipitation": 0, "windSpeed": 12},
        "occasion": "working from home",
        "theme": "comfortable but presentable for video calls",
        "timeOfDay": "morning",
        "dayOfWeek": "Wednesday",
    },
    {
        "id": "humid_wedding",
        "weather": {"temperature": 28, "condition": "Humid", "precipitation": 0, "windSpeed": 4},
        "occasion": "outdoor wedding",
        "theme": None,
        "timeOfDay": "afternoon",
        "dayOfWeek": "Sunday",
    },
    {
        "id": "windy_outdoor_walk",
        "weather": {"temperature": 10, "condition": "Windy", "precipitation": 0, "windSpeed": 35},
        "occasion": "long walk in the park",
        "theme": None,
        "timeOfDay": "midday",
        "dayOfWeek": "Sunday",
    },
    {
        "id": "cool_workout",
        "weather": {"temperature": 15, "condition": "Clear", "precipitation": 0, "windSpeed": 6},
        "occasion": "outdoor run",
        "theme": None,
        "timeOfDay": "morning",
        "dayOfWeek": "Thursday",
    },
    {
        "id": "cold_evening_out",
        "weather": {"temperature": 6, "condition": "Clear", "precipitation": 0, "windSpeed": 10},
        "occasion": "concert",
        "theme": "want to stand out a bit",
        "timeOfDay": "evening",
        "dayOfWeek": "Friday",
    },
]

pd.DataFrame(FIXTURES)


## 5. Shared helpers — fixture state + context text


In [ ]:
def fixture_state(fx: dict) -> dict:
    """Structured state dict — accepted directly by Laya + OpenDecision.
    For PrismNLI we json.dumps this into the premise text."""
    w = fx["weather"]
    return {
        "weather": {
            "temperature_c": w["temperature"],
            "condition": w["condition"],
            "precipitation_mm": w.get("precipitation", 0),
            "wind_kmh": w.get("windSpeed", 0),
        },
        "occasion": fx.get("occasion"),
        "time_of_day": fx.get("timeOfDay"),
        "day_of_week": fx.get("dayOfWeek"),
        "theme": fx.get("theme"),
    }


def fixture_context_text(fx: dict) -> str:
    """Natural-language context — used as PrismNLI's premise + as the
    filler in NLI hypothesis templates."""
    w = fx["weather"]
    parts = [
        f"{fx.get('timeOfDay') or 'day'} on {fx.get('dayOfWeek') or 'a weekday'}",
        f"weather is {w['condition']} at {w['temperature']}°C",
    ]
    if w.get("precipitation", 0) > 0:
        parts.append(f"precipitation {w['precipitation']} mm")
    if w.get("windSpeed", 0) > 15:
        parts.append(f"windy {w['windSpeed']} km/h")
    if fx.get("occasion"):
        parts.append(f"occasion is {fx['occasion']}")
    if fx.get("theme"):
        parts.append(f"note: {fx['theme']}")
    return "; ".join(parts)


CHOICE_MAX_OPTIONS = 20  # Laya card limit; also a reasonable soft cap for the NLI candidates


## 6a. Laya inference helpers — Choice + Noul (unchanged from single-vendor benchmark)

Mirrors nb20's Jev call shape exactly so downstream metrics reuse the
same harness. Cost is 0 (self-hosted). Tokens best-effort.

**Choice chunking:** the Laya card's *Limits* section says
*"Keep `choice` questions under ~20 options."* If our closet exceeds 20,
we split, take each half's winner, run a final tie-break Choice on the
two winners.

In [ ]:
def _laya_predict(state, questions, retries: int = 1):
    """Thin wrapper — retry once on transient error, return (result, latency_s, err)."""
    last_err = None
    for attempt in range(retries + 1):
        t0 = time.perf_counter()
        try:
            res = laya_agent.predict(state, questions)
            return res, time.perf_counter() - t0, None
        except Exception as e:
            last_err = f"{type(e).__name__}: {e}"
            if attempt < retries:
                time.sleep(0.5)
                continue
    return None, 0.0, last_err


def laya_choice(fx: dict, outfits: list[dict], by_id: dict[str, dict]) -> dict:
    """Single Choice over all outfits, chunked if > CHOICE_MAX_OPTIONS."""
    criteria_all = outfit_criteria(outfits, by_id)
    state = fixture_state(fx)

    all_probs: dict[str, float] = {}
    total_latency = 0.0
    first_err = None
    n_calls = 0

    outfit_ids = [o["id"] for o in outfits if o["id"] in criteria_all]

    def _run_choice(sub_ids: list[str]) -> dict | None:
        nonlocal total_latency, first_err, n_calls
        sub_criteria = {oid: criteria_all[oid] for oid in sub_ids}
        qs = {
            "best": {
                "type": "choice",
                "instructions": (
                    "Pick the single best saved outfit for the given weather + "
                    "occasion + time context."
                ),
                "criteria": sub_criteria,
            },
        }
        res, lat, err = _laya_predict(state, qs)
        total_latency += lat
        n_calls += 1
        if err:
            first_err = first_err or err
            return None
        return (res.get("answers") or {}).get("best") or None

    if len(outfit_ids) <= CHOICE_MAX_OPTIONS:
        ans = _run_choice(outfit_ids)
        top_id = ans.get("choice") if ans else None
        confidence = ans.get("confidence") if ans else None
        probabilities = ans.get("probabilities") or {} if ans else {}
    else:
        mid = len(outfit_ids) // 2
        halves = [outfit_ids[:mid], outfit_ids[mid:]]
        half_winners: list[str] = []
        half_probs: list[dict[str, float]] = []
        for half in halves:
            ans = _run_choice(half)
            if ans and ans.get("choice"):
                half_winners.append(ans["choice"])
                half_probs.append(dict(ans.get("probabilities") or {}))

        for pmap in half_probs:
            all_probs.update(pmap)

        if len(half_winners) >= 2:
            tie = _run_choice(half_winners)
            top_id = tie.get("choice") if tie else half_winners[0]
            confidence = tie.get("confidence") if tie else None
            all_probs.update(dict((tie or {}).get("probabilities") or {}))
            probabilities = all_probs
        elif half_winners:
            top_id = half_winners[0]
            confidence = None
            probabilities = all_probs
        else:
            top_id = None
            confidence = None
            probabilities = {}

    return {
        "fixture_id": fx["id"],
        "provider": "laya-choice",
        "model": LAYA_CHECKPOINT,
        "ok": first_err is None,
        "error": first_err,
        "top_id": top_id,
        "confidence": confidence,
        "probabilities": probabilities,
        "latency_s": total_latency,
        "n_calls": n_calls,
        "tokens_in": 0,     # TODO: verify — populate from res["usage"] if present
        "tokens_out": 0,
        "cost_usd": 0.0,
    }


def laya_noul(fx: dict, outfits: list[dict], by_id: dict[str, dict]) -> dict:
    """One batched Noul call fanning out per outfit."""
    state = fixture_state(fx)
    criteria_map = outfit_criteria(outfits, by_id)
    questions = {
        f"q_{o['id']}": {
            "type": "noul",
            "instructions": f"Is outfit '{criteria_map[o['id']]}' a good pick for the given context?",
        }
        for o in outfits if o["id"] in criteria_map
    }
    res, lat, err = _laya_predict(state, questions)
    answers = (res or {}).get("answers") or {}

    ranked: list[tuple[str, float]] = []
    for o in outfits:
        a = answers.get(f"q_{o['id']}") or {}
        nv = a.get("noul")
        if nv is not None:
            ranked.append((o["id"], float(nv)))
    ranked.sort(key=lambda kv: kv[1], reverse=True)

    return {
        "fixture_id": fx["id"],
        "provider": "laya-noul",
        "model": LAYA_CHECKPOINT,
        "ok": err is None,
        "error": err,
        "ranked_ids": [oid for oid, _ in ranked],
        "top_id": ranked[0][0] if ranked else None,
        "top_score": ranked[0][1] if ranked else None,
        "all_scores": {oid: s for oid, s in ranked},
        "latency_s": lat,
        "tokens_in": 0,
        "tokens_out": 0,
        "cost_usd": 0.0,
    }


## 6b. PrismNLI inference helpers — Choice + Noul via zero-shot pipeline

**Choice adapter:** the DeBERTa zero-shot pipeline takes a `premise` string
plus `candidate_labels`. We use the natural-language context text
(§5 `fixture_context_text`) as premise, and *the outfit descriptions
themselves* as candidate labels — this is how the pipeline was designed
per the model card. Hypothesis template: `"This outfit is the best pick
for the given context: {}"`. Pipeline returns `{labels, scores}` where
scores are entailment probabilities summing to 1.

We build a reverse map `{description → outfit_id}` so we can translate
back to Jev's `probabilities: {outfit_id: prob}` shape.

**Noul adapter:** per-outfit binary classification. For outfit `o`, run
`pipeline(premise=context, candidate_labels=["yes","no"],
hypothesis_template=f"This outfit '{{}}' — {desc} — is a good pick for
the current context.")`. Take `yes` score as noul value.

**Chunking:** same 20-cap as Laya.

In [ ]:
CHOICE_HYPOTHESIS_TEMPLATE = "This outfit is the best pick for the given context: {}"
NOUL_HYPOTHESIS_TEMPLATE = "This outfit is a good pick for the given context: {}"


def _prismnli_predict(premise: str, candidate_labels: list[str], hypothesis_template: str,
                       multi_label: bool = False, retries: int = 1):
    last_err = None
    for attempt in range(retries + 1):
        t0 = time.perf_counter()
        try:
            res = prismnli_pipe(
                premise,
                candidate_labels=candidate_labels,
                hypothesis_template=hypothesis_template,
                multi_label=multi_label,
            )
            return res, time.perf_counter() - t0, None
        except Exception as e:
            last_err = f"{type(e).__name__}: {e}"
            if attempt < retries:
                time.sleep(0.5)
                continue
    return None, 0.0, last_err


def prismnli_choice(fx: dict, outfits: list[dict], by_id: dict[str, dict]) -> dict:
    """Zero-shot Choice — outfit descriptions become candidate labels."""
    criteria_all = outfit_criteria(outfits, by_id)
    premise = fixture_context_text(fx)
    outfit_ids = [o["id"] for o in outfits if o["id"] in criteria_all]

    total_latency = 0.0
    first_err = None
    n_calls = 0
    all_probs: dict[str, float] = {}

    def _run(sub_ids: list[str]) -> tuple[str | None, float | None, dict[str, float]]:
        nonlocal total_latency, first_err, n_calls
        # Descriptions become labels; we translate back via desc → id map.
        desc_to_id = {criteria_all[oid]: oid for oid in sub_ids}
        labels = list(desc_to_id.keys())
        res, lat, err = _prismnli_predict(
            premise=premise,
            candidate_labels=labels,
            hypothesis_template=CHOICE_HYPOTHESIS_TEMPLATE,
            multi_label=False,
        )
        total_latency += lat
        n_calls += 1
        if err:
            first_err = first_err or err
            return None, None, {}
        # Pipeline returns labels sorted by score descending.
        top_label = res["labels"][0]
        top_score = float(res["scores"][0])
        probs = {desc_to_id[lbl]: float(sc) for lbl, sc in zip(res["labels"], res["scores"])}
        return desc_to_id[top_label], top_score, probs

    if len(outfit_ids) <= CHOICE_MAX_OPTIONS:
        top_id, confidence, probabilities = _run(outfit_ids)
    else:
        mid = len(outfit_ids) // 2
        halves = [outfit_ids[:mid], outfit_ids[mid:]]
        half_winners: list[str] = []
        for half in halves:
            winner, _score, half_probs = _run(half)
            if winner:
                half_winners.append(winner)
                all_probs.update(half_probs)
        if len(half_winners) >= 2:
            top_id, confidence, tie_probs = _run(half_winners)
            all_probs.update(tie_probs)
            probabilities = all_probs
        elif half_winners:
            top_id = half_winners[0]
            confidence = None
            probabilities = all_probs
        else:
            top_id = None
            confidence = None
            probabilities = {}

    return {
        "fixture_id": fx["id"],
        "provider": "prismnli-choice",
        "model": PRISMNLI_CHECKPOINT,
        "ok": first_err is None,
        "error": first_err,
        "top_id": top_id,
        "confidence": confidence,
        "probabilities": probabilities,
        "latency_s": total_latency,
        "n_calls": n_calls,
        "tokens_in": 0,
        "tokens_out": 0,
        "cost_usd": 0.0,
    }


def prismnli_noul(fx: dict, outfits: list[dict], by_id: dict[str, dict]) -> dict:
    """Per-outfit binary NLI classification, taking entailment('yes') as noul."""
    premise = fixture_context_text(fx)
    criteria_map = outfit_criteria(outfits, by_id)

    total_latency = 0.0
    first_err = None
    scores: dict[str, float] = {}

    for o in outfits:
        if o["id"] not in criteria_map:
            continue
        desc = criteria_map[o["id"]]
        # TODO: verify — could also batch these via pipeline([premise]*N, ...)
        # but pipeline handles single-example better; measure both if latency
        # becomes a bottleneck.
        res, lat, err = _prismnli_predict(
            premise=premise,
            candidate_labels=["yes", "no"],
            hypothesis_template=NOUL_HYPOTHESIS_TEMPLATE.replace("{}", desc + ": {}"),
            multi_label=False,
        )
        total_latency += lat
        if err:
            first_err = first_err or err
            continue
        # Pipeline returns labels sorted by score; find "yes" entry.
        yes_score = next(
            (float(s) for lbl, s in zip(res["labels"], res["scores"]) if lbl == "yes"),
            None,
        )
        if yes_score is not None:
            scores[o["id"]] = yes_score

    ranked = sorted(scores.items(), key=lambda kv: kv[1], reverse=True)

    return {
        "fixture_id": fx["id"],
        "provider": "prismnli-noul",
        "model": PRISMNLI_CHECKPOINT,
        "ok": first_err is None,
        "error": first_err,
        "ranked_ids": [oid for oid, _ in ranked],
        "top_id": ranked[0][0] if ranked else None,
        "top_score": ranked[0][1] if ranked else None,
        "all_scores": dict(ranked),
        "latency_s": total_latency,
        "tokens_in": 0,
        "tokens_out": 0,
        "cost_usd": 0.0,
    }


## 6c. OpenDecision inference helpers — Choice + Noul (native primitives)

OpenDecision's `engine.choice()` / `engine.noul()` have the same
call shape as Jev / Laya:

```python
engine.choice(state=..., instructions=..., criteria={id: desc, ...})
engine.noul(state=..., instructions=...)
```

so translation is 1:1 — no NLI hypothesis engineering needed here.

**Choice chunking:** OpenDecision docs don't specify a candidate cap;
its two-compiler + adjudicator design (`_choice_profile` in
`src/opendecision/engine.py`) is more robust to many-way choices than
raw NLI, but we still apply the 20-cap for latency parity with Laya
and PrismNLI. Adjust `CHOICE_MAX_OPTIONS` if you want to A/B.

In [ ]:
def _od_call(fn, retries: int = 1, **kwargs):
    last_err = None
    for attempt in range(retries + 1):
        t0 = time.perf_counter()
        try:
            res = fn(**kwargs)
            return res, time.perf_counter() - t0, None
        except Exception as e:
            last_err = f"{type(e).__name__}: {e}"
            if attempt < retries:
                time.sleep(0.5)
                continue
    return None, 0.0, last_err


def opendecision_choice(fx: dict, outfits: list[dict], by_id: dict[str, dict]) -> dict:
    criteria_all = outfit_criteria(outfits, by_id)
    state = fixture_state(fx)
    outfit_ids = [o["id"] for o in outfits if o["id"] in criteria_all]

    total_latency = 0.0
    first_err = None
    n_calls = 0
    all_probs: dict[str, float] = {}

    instructions = (
        "Pick the single best saved outfit for the given weather + "
        "occasion + time context."
    )

    def _run(sub_ids: list[str]) -> dict | None:
        nonlocal total_latency, first_err, n_calls
        sub_criteria = {oid: criteria_all[oid] for oid in sub_ids}
        res, lat, err = _od_call(
            opendecision_engine.choice,
            state=state,
            instructions=instructions,
            criteria=sub_criteria,
        )
        total_latency += lat
        n_calls += 1
        if err:
            first_err = first_err or err
            return None
        return res

    if len(outfit_ids) <= CHOICE_MAX_OPTIONS:
        ans = _run(outfit_ids)
        top_id = ans.get("choice") if ans else None
        confidence = ans.get("confidence") if ans else None
        probabilities = dict(ans.get("probabilities") or {}) if ans else {}
    else:
        mid = len(outfit_ids) // 2
        halves = [outfit_ids[:mid], outfit_ids[mid:]]
        half_winners: list[str] = []
        for half in halves:
            ans = _run(half)
            if ans and ans.get("choice"):
                half_winners.append(ans["choice"])
                all_probs.update(dict(ans.get("probabilities") or {}))
        if len(half_winners) >= 2:
            tie = _run(half_winners)
            top_id = tie.get("choice") if tie else half_winners[0]
            confidence = tie.get("confidence") if tie else None
            all_probs.update(dict((tie or {}).get("probabilities") or {}))
            probabilities = all_probs
        elif half_winners:
            top_id = half_winners[0]
            confidence = None
            probabilities = all_probs
        else:
            top_id = None
            confidence = None
            probabilities = {}

    return {
        "fixture_id": fx["id"],
        "provider": "opendecision-choice",
        "model": "MoritzLaurer/ModernBERT-large-zeroshot-v2.0",
        "ok": first_err is None,
        "error": first_err,
        "top_id": top_id,
        "confidence": confidence,
        "probabilities": probabilities,
        "latency_s": total_latency,
        "n_calls": n_calls,
        "tokens_in": 0,
        "tokens_out": 0,
        "cost_usd": 0.0,
    }


def opendecision_noul(fx: dict, outfits: list[dict], by_id: dict[str, dict]) -> dict:
    state = fixture_state(fx)
    criteria_map = outfit_criteria(outfits, by_id)

    total_latency = 0.0
    first_err = None
    scores: dict[str, float] = {}

    for o in outfits:
        if o["id"] not in criteria_map:
            continue
        desc = criteria_map[o["id"]]
        # OpenDecision noul(): predicate form — hypothesis is `instructions`.
        # TODO: verify — batching-friendly API not documented; if latency
        # becomes a bottleneck, contribute a batch endpoint upstream.
        res, lat, err = _od_call(
            opendecision_engine.noul,
            state=state,
            instructions=f"This outfit is a good pick for the current context: {desc}.",
        )
        total_latency += lat
        if err:
            first_err = first_err or err
            continue
        nv = res.get("noul") if res else None
        if nv is not None:
            scores[o["id"]] = float(nv)

    ranked = sorted(scores.items(), key=lambda kv: kv[1], reverse=True)

    return {
        "fixture_id": fx["id"],
        "provider": "opendecision-noul",
        "model": "MoritzLaurer/ModernBERT-large-zeroshot-v2.0",
        "ok": first_err is None,
        "error": first_err,
        "ranked_ids": [oid for oid, _ in ranked],
        "top_id": ranked[0][0] if ranked else None,
        "top_score": ranked[0][1] if ranked else None,
        "all_scores": dict(ranked),
        "latency_s": total_latency,
        "tokens_in": 0,
        "tokens_out": 0,
        "cost_usd": 0.0,
    }


## 7. Run all fixtures through every enabled candidate

One Choice + one Noul per fixture per candidate. Raw outputs cached to
`prototypes/cache/22_<candidate>_{choice,noul}_raw.json` so the metric
cells can be re-run without rehitting the models.

In [ ]:
CANDIDATE_RUNNERS = {
    "laya": (RUN_LAYA, laya_choice if RUN_LAYA else None, laya_noul if RUN_LAYA else None),
    "prismnli": (RUN_PRISMNLI, prismnli_choice if RUN_PRISMNLI else None, prismnli_noul if RUN_PRISMNLI else None),
    "opendecision": (RUN_OPENDECISION, opendecision_choice if RUN_OPENDECISION else None, opendecision_noul if RUN_OPENDECISION else None),
}

candidate_results: dict[str, dict[str, list[dict]]] = {
    name: {"choice": [], "noul": []} for name in CANDIDATE_RUNNERS
}

if len(outfits) < 2:
    print(f"SKIP: need at least 2 outfits, have {len(outfits)}.")
else:
    for cand, (enabled, choice_fn, noul_fn) in CANDIDATE_RUNNERS.items():
        if not enabled:
            print(f"\n== {cand.upper()} — disabled, skipping ==")
            continue
        print(f"\n== {cand.upper()} ==")
        for fx in FIXTURES:
            rc = choice_fn(fx, outfits, items_by_id)
            candidate_results[cand]["choice"].append(rc)
            print(
                f"[{cand[:5]}-c] {fx['id']:32s}  top={rc['top_id']}  "
                f"conf={rc['confidence']}  lat={rc['latency_s']:.3f}s  "
                f"calls={rc.get('n_calls')}  err={rc['error']}"
            )
            rn = noul_fn(fx, outfits, items_by_id)
            candidate_results[cand]["noul"].append(rn)
            print(
                f"[{cand[:5]}-n] {fx['id']:32s}  top={rn['top_id']}  "
                f"score={rn['top_score']}  lat={rn['latency_s']:.3f}s  err={rn['error']}"
            )

        # Persist per-candidate raw outputs
        (CACHE_DIR / f"22_{cand}_choice_raw.json").write_text(
            json.dumps(candidate_results[cand]["choice"], indent=2, default=str)
        )
        (CACHE_DIR / f"22_{cand}_noul_raw.json").write_text(
            json.dumps(candidate_results[cand]["noul"], indent=2, default=str)
        )
        print(f"  Wrote {CACHE_DIR}/22_{cand}_choice_raw.json + 22_{cand}_noul_raw.json")


## 8. Comparison metrics — per-candidate × per-gate table vs cached Jev

Loads `20_jev_choice_raw.json` + `20_jev_noul_raw.json` verbatim.
Per fixture per candidate:
- **Choice agreement:** does candidate top-1 = Jev top-1?
- **Noul rank correlation:** Spearman ρ + Kendall τ vs Jev's noul scores.
- **Latency:** median + p90 per candidate.
- **Confidence:** distribution vs Jev's.
- **ECE (noul):** expected calibration error using Jev's noul as
  ground-truth probability, 10 bins.

We compute Spearman/Kendall locally (no scipy dependency).

In [ ]:
def _rank(values: list[float]) -> list[float]:
    ordered = sorted(range(len(values)), key=lambda i: values[i])
    ranks = [0.0] * len(values)
    i = 0
    while i < len(ordered):
        j = i
        while j + 1 < len(ordered) and values[ordered[j + 1]] == values[ordered[i]]:
            j += 1
        avg = (i + j) / 2 + 1
        for k in range(i, j + 1):
            ranks[ordered[k]] = avg
        i = j + 1
    return ranks


def spearman(xs: list[float], ys: list[float]) -> float | None:
    if len(xs) != len(ys) or len(xs) < 2:
        return None
    rx = _rank(xs)
    ry = _rank(ys)
    n = len(xs)
    mean_rx = sum(rx) / n
    mean_ry = sum(ry) / n
    num = sum((rx[i] - mean_rx) * (ry[i] - mean_ry) for i in range(n))
    den_x = sum((r - mean_rx) ** 2 for r in rx) ** 0.5
    den_y = sum((r - mean_ry) ** 2 for r in ry) ** 0.5
    if den_x == 0 or den_y == 0:
        return None
    return num / (den_x * den_y)


def kendall_tau(xs: list[float], ys: list[float]) -> float | None:
    if len(xs) != len(ys) or len(xs) < 2:
        return None
    n = len(xs)
    concordant = discordant = 0
    for i in range(n):
        for j in range(i + 1, n):
            dx = xs[i] - xs[j]
            dy = ys[i] - ys[j]
            s = dx * dy
            if s > 0:
                concordant += 1
            elif s < 0:
                discordant += 1
    denom = n * (n - 1) / 2
    if denom == 0:
        return None
    return (concordant - discordant) / denom


def ece(pred: list[float], truth: list[float], n_bins: int = 10) -> float | None:
    """Expected calibration error, using Jev's noul as ground-truth probability.
    Bins predictions into n_bins equal-width bins on [0,1] and compares each
    bin's mean predicted probability against the mean truth probability."""
    if len(pred) != len(truth) or not pred:
        return None
    bins = [[] for _ in range(n_bins)]
    for p, t in zip(pred, truth):
        b = min(n_bins - 1, max(0, int(p * n_bins)))
        bins[b].append((p, t))
    total = len(pred)
    total_err = 0.0
    for bucket in bins:
        if not bucket:
            continue
        mp = sum(p for p, _ in bucket) / len(bucket)
        mt = sum(t for _, t in bucket) / len(bucket)
        total_err += (len(bucket) / total) * abs(mp - mt)
    return total_err


# Load Jev cache
JEV_CHOICE = json.loads((CACHE_DIR / "20_jev_choice_raw.json").read_text())
JEV_NOUL = json.loads((CACHE_DIR / "20_jev_noul_raw.json").read_text())
jev_c_by_fx = {r["fixture_id"]: r for r in JEV_CHOICE}
jev_n_by_fx = {r["fixture_id"]: r for r in JEV_NOUL}


def build_metrics_df(cand: str) -> pd.DataFrame:
    c_by_fx = {r["fixture_id"]: r for r in candidate_results[cand]["choice"]}
    n_by_fx = {r["fixture_id"]: r for r in candidate_results[cand]["noul"]}
    rows = []
    for fx in FIXTURES:
        fid = fx["id"]
        jc = jev_c_by_fx.get(fid, {})
        jn = jev_n_by_fx.get(fid, {})
        cc = c_by_fx.get(fid, {})
        cn = n_by_fx.get(fid, {})

        jev_scores = jn.get("all_scores") or {}
        cand_scores = cn.get("all_scores") or {}
        shared_ids = [oid for oid in jev_scores if oid in cand_scores]
        xs = [float(jev_scores[oid]) for oid in shared_ids]
        ys = [float(cand_scores[oid]) for oid in shared_ids]

        rows.append({
            "candidate": cand,
            "fixture_id": fid,
            # Choice
            "jev_choice_top": jc.get("top_id"),
            "jev_choice_conf": jc.get("confidence"),
            "cand_choice_top": cc.get("top_id"),
            "cand_choice_conf": cc.get("confidence"),
            "choice_agree": (jc.get("top_id") == cc.get("top_id")) if (jc.get("top_id") and cc.get("top_id")) else None,
            # Noul
            "jev_noul_top": jn.get("top_id"),
            "cand_noul_top": cn.get("top_id"),
            "noul_top_agree": (jn.get("top_id") == cn.get("top_id")) if (jn.get("top_id") and cn.get("top_id")) else None,
            "noul_spearman_rho": spearman(xs, ys),
            "noul_kendall_tau": kendall_tau(xs, ys),
            "noul_ece": ece(ys, xs),  # candidate = pred, jev = truth
            "noul_shared_n": len(shared_ids),
            # Latency
            "jev_choice_lat_s": jc.get("latency_s"),
            "jev_noul_lat_s": jn.get("latency_s"),
            "cand_choice_lat_s": cc.get("latency_s"),
            "cand_noul_lat_s": cn.get("latency_s"),
            # Cost
            "jev_cost_usd": (jc.get("cost_usd") or 0) + (jn.get("cost_usd") or 0),
            "cand_cost_usd": 0.0,
        })
    return pd.DataFrame(rows)


all_metrics = pd.concat(
    [build_metrics_df(cand) for cand, (enabled, _, _) in CANDIDATE_RUNNERS.items() if enabled],
    ignore_index=True,
) if any(enabled for enabled, _, _ in CANDIDATE_RUNNERS.values()) else pd.DataFrame()

csv_path = OUT_DIR / "22_metrics.csv"
all_metrics.to_csv(csv_path, index=False)
print(f"Wrote {csv_path}  ({len(all_metrics)} rows across candidates)")
all_metrics


## 9. Aggregate — per-candidate summary + gate table


In [ ]:
def _pct(s):
    return s.mean() if len(s) else None


def _median(s):
    return float(s.median()) if len(s) else None


def _p90(s):
    if not len(s):
        return None
    vals = sorted(s.tolist())
    return float(vals[int(0.9 * (len(vals) - 1))])


gate_rows = []
if all_metrics.empty:
    print("No candidate metrics available — skip aggregation.")
else:
    for cand in all_metrics["candidate"].unique():
        cdf = all_metrics[all_metrics["candidate"] == cand]

        choice_agree = cdf["choice_agree"].dropna()
        noul_top_agree = cdf["noul_top_agree"].dropna()
        rho = cdf["noul_spearman_rho"].dropna().astype(float)
        tau = cdf["noul_kendall_tau"].dropna().astype(float)
        ece_vals = cdf["noul_ece"].dropna().astype(float)

        c_lat = cdf["cand_choice_lat_s"].dropna().astype(float)
        n_lat = cdf["cand_noul_lat_s"].dropna().astype(float)

        jev_conf = cdf["jev_choice_conf"].dropna().astype(float)
        cand_conf = cdf["cand_choice_conf"].dropna().astype(float)

        agree_mean = _pct(choice_agree)
        c_lat_med = _median(c_lat)
        n_lat_med = _median(n_lat)
        conf_delta = (
            abs(_median(jev_conf) - _median(cand_conf))
            if len(jev_conf) and len(cand_conf)
            else None
        )
        ece_med = _median(ece_vals)

        gate_agree = agree_mean is not None and agree_mean >= 0.90
        gate_lat = (
            c_lat_med is not None and c_lat_med < 1.0
            and (n_lat_med is None or n_lat_med < 1.0)
        )
        gate_conf = conf_delta is not None and conf_delta <= 0.05
        all_pass = bool(gate_agree and gate_lat and gate_conf)

        print(f"\n── {cand.upper()} ──")
        print(f"Choice agreement (vs Jev top-1):        {agree_mean:.0%}  (n={len(choice_agree)})" if agree_mean is not None else "Choice agreement: n/a")
        print(f"Noul top-1 agreement:                   " + (f"{_pct(noul_top_agree):.0%}  (n={len(noul_top_agree)})" if _pct(noul_top_agree) is not None else "n/a"))
        if len(rho):
            print(f"Noul Spearman ρ (median):               {rho.median():.3f}   [{rho.min():.3f}, {rho.max():.3f}]")
        if len(tau):
            print(f"Noul Kendall τ (median):                {tau.median():.3f}")
        if ece_med is not None:
            print(f"Noul ECE (median, 10-bin, Jev as truth): {ece_med:.3f}")
        print(f"Latency (Choice)  median / p90:         {c_lat_med:.3f}s / {_p90(c_lat):.3f}s" if c_lat_med is not None else "Latency Choice: n/a")
        print(f"Latency (Noul)    median / p90:         {n_lat_med:.3f}s / {_p90(n_lat):.3f}s" if n_lat_med is not None else "Latency Noul: n/a")
        if conf_delta is not None:
            print(f"Confidence median delta vs Jev:         {conf_delta:.3f}")

        print(f"  Gate 1 (agreement >= 90%):            {'PASS' if gate_agree else 'FAIL'}")
        print(f"  Gate 2 (choice + noul median < 1s):   {'PASS' if gate_lat else 'FAIL'}")
        print(f"  Gate 3 (conf median delta <= 0.05):   {'PASS' if gate_conf else 'FAIL'}")
        print(f"  → ALL PASS: {all_pass}")

        gate_rows.append({
            "candidate": cand,
            "choice_agreement": agree_mean,
            "noul_top_agreement": _pct(noul_top_agree),
            "noul_spearman_median": (float(rho.median()) if len(rho) else None),
            "noul_ece_median": ece_med,
            "choice_lat_median_s": c_lat_med,
            "choice_lat_p90_s": _p90(c_lat),
            "noul_lat_median_s": n_lat_med,
            "noul_lat_p90_s": _p90(n_lat),
            "conf_median_delta": conf_delta,
            "gate_agreement": bool(gate_agree),
            "gate_latency": bool(gate_lat),
            "gate_confidence": bool(gate_conf),
            "all_gates_pass": all_pass,
        })

gates_df = pd.DataFrame(gate_rows)
if not gates_df.empty:
    gates_path = OUT_DIR / "22_gates.csv"
    gates_df.to_csv(gates_path, index=False)
    print(f"\nWrote {gates_path}")
gates_df


## 10. Winner selection

**Rules (from the epic + plan doc):**
1. Filter to candidates where `all_gates_pass = True`.
2. If exactly one → that's the winner.
3. If multiple → maximise `choice_agreement × (1 − noul_ece_median)`.
4. Tie-break on deployment simplicity:
   `laya (pip, py>=3.9)` > `prismnli (pip, py>=3.9)` > `opendecision (pip, py>=3.13)`.
5. If none pass → recommend keeping Jev; report the *closest* candidate
   and which gate(s) it missed.

In [ ]:
DEPLOY_SIMPLICITY_ORDER = ["laya", "prismnli", "opendecision"]


def _simplicity_rank(name: str) -> int:
    return DEPLOY_SIMPLICITY_ORDER.index(name) if name in DEPLOY_SIMPLICITY_ORDER else 99


if gates_df.empty:
    print("No gates data — cannot select winner.")
else:
    passing = gates_df[gates_df["all_gates_pass"]].copy()
    if passing.empty:
        print("── NO CANDIDATE PASSES ALL GATES ──")
        print("Recommendation: KEEP JEV.")
        # Report the closest — sort by gates_passed desc, then agreement desc.
        gates_df["gates_passed_count"] = (
            gates_df["gate_agreement"].astype(int)
            + gates_df["gate_latency"].astype(int)
            + gates_df["gate_confidence"].astype(int)
        )
        closest = gates_df.sort_values(
            ["gates_passed_count", "choice_agreement"], ascending=[False, False]
        ).iloc[0]
        missed = []
        if not closest["gate_agreement"]:
            missed.append(f"agreement={closest['choice_agreement']:.2%} (<90%)")
        if not closest["gate_latency"]:
            missed.append(f"latency={closest['choice_lat_median_s']:.3f}s (>=1s)")
        if not closest["gate_confidence"]:
            missed.append(f"conf_delta={closest['conf_median_delta']:.3f} (>0.05)")
        print(f"Closest miss: {closest['candidate'].upper()} — failed {', '.join(missed)}")
    else:
        passing["quality_score"] = passing["choice_agreement"] * (
            1 - passing["noul_ece_median"].fillna(0.0)
        )
        passing["simplicity_rank"] = passing["candidate"].map(_simplicity_rank)
        passing = passing.sort_values(
            ["quality_score", "simplicity_rank"], ascending=[False, True]
        )

        print("── PASSING CANDIDATES (sorted by quality × simplicity) ──")
        print(passing[[
            "candidate", "choice_agreement", "noul_ece_median",
            "quality_score", "simplicity_rank",
            "choice_lat_median_s", "noul_lat_median_s",
        ]].to_string(index=False))

        winner = passing.iloc[0]
        print(f"\n🏆 WINNER: {winner['candidate'].upper()}")
        print(f"   choice_agreement = {winner['choice_agreement']:.2%}")
        print(f"   noul_ece         = {winner['noul_ece_median']}")
        print(f"   quality_score    = {winner['quality_score']:.4f}")
        print(f"   deploy_rank      = {int(winner['simplicity_rank'])} (lower is simpler)")

        (OUT_DIR / "22_winner.json").write_text(json.dumps({
            "winner": winner["candidate"],
            "choice_agreement": float(winner["choice_agreement"]),
            "noul_ece": float(winner["noul_ece_median"]) if winner["noul_ece_median"] is not None else None,
            "quality_score": float(winner["quality_score"]),
        }, indent=2))
        print(f"\nWrote {OUT_DIR / '22_winner.json'}")


## 11. Charts — confidence overlay + per-candidate latency bar


In [ ]:
if not all_metrics.empty:
    candidates = list(all_metrics["candidate"].unique())
    n = len(candidates)

    fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))

    # Confidence overlay — Jev + each candidate
    bins = [i / 10 for i in range(11)]
    jev_conf_vals = all_metrics.groupby("fixture_id").first()["jev_choice_conf"].dropna().astype(float).tolist()
    if jev_conf_vals:
        axes[0].hist(jev_conf_vals, bins=bins, alpha=0.4, label=f"Jev (n={len(jev_conf_vals)})", edgecolor="black")
    palette = ["#4c72b0", "#dd8452", "#55a868", "#c44e52"]
    for i, cand in enumerate(candidates):
        vals = all_metrics[all_metrics["candidate"] == cand]["cand_choice_conf"].dropna().astype(float).tolist()
        if vals:
            axes[0].hist(vals, bins=bins, alpha=0.45, label=f"{cand} (n={len(vals)})",
                         edgecolor="black", color=palette[i % len(palette)])
    axes[0].set_xlabel("Choice top-1 confidence")
    axes[0].set_ylabel("Count")
    axes[0].set_title("Confidence distribution — Jev vs. candidates")
    axes[0].legend(fontsize=8)

    # Latency bar — median per candidate per primitive
    labels = []
    medians = []
    colors = []
    color_map = {c: palette[i % len(palette)] for i, c in enumerate(candidates)}
    for cand in candidates:
        cdf = all_metrics[all_metrics["candidate"] == cand]
        for prim, col in [("Choice", "cand_choice_lat_s"), ("Noul", "cand_noul_lat_s")]:
            vals = cdf[col].dropna().astype(float)
            labels.append(f"{cand}\n{prim}")
            medians.append(float(vals.median()) if len(vals) else 0.0)
            colors.append(color_map[cand])
    bars = axes[1].bar(labels, medians, color=colors)
    axes[1].axhline(1.0, color="red", linestyle="--", label="Latency gate (1s)")
    axes[1].set_ylabel("Median latency (s)")
    axes[1].set_title("Median call latency per candidate × primitive")
    axes[1].tick_params(axis="x", rotation=15)
    axes[1].legend(fontsize=8)
    for bar, val in zip(bars, medians):
        axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
                     f"{val:.2f}s", ha="center", fontsize=7)

    plt.tight_layout()
    png_path = OUT_DIR / "22_summary.png"
    plt.savefig(png_path, dpi=100)
    plt.show()
    print(f"Saved chart to {png_path}")


## 12. Interpretation + recommendation

Fill in numerically after the run — cells 9 + 10 print PASS/FAIL per gate
and select a winner (or "keep Jev" if none pass).

**Vendor-claim latency baselines to sanity-check against your run:**

| Candidate | CPU per-decision (claimed) | GPU (T4, claimed) |
|---|---|---|
| Laya | 193–464 ms | ~40 ms |
| PrismNLI (DeBERTa-v3-large) | Not documented; expect ~500–1500 ms per NLI pass at 512 tokens | ~80 ms |
| OpenDecision (ModernBERT-large) | Not documented; ModernBERT is ~2× faster than DeBERTa-v3, so expect ~250–800 ms | ~40 ms |

**Most-likely-to-fail-latency-gate flag:** PrismNLI's Noul primitive is
`N per-outfit calls` (~30 outfits × ~800 ms each = ~24 s per fixture on
CPU). If Aryan's kernel doesn't have a T4, the PrismNLI Noul gate is
likely to FAIL — but this is a *harness* limitation (per-outfit-call
overhead), not a model limitation. Batching the pipeline into a single
`pipe([premise]*N, ...)` call could recover ~5–10× on modern
transformers. Marked as `# TODO: verify` in cell 6b.

**If Laya wins → same as before:** self-host on the Render dyno, zero
marginal per-call cost, cleaner GDPR posture, no OpenRouter alpha
endpoint dependency.

**If PrismNLI wins →** MIT-licensed (more permissive than Apache 2.0 for
some downstream users), well-known checkpoint with 400M-user
verification on HF, standard transformers pipeline (no bespoke SDK).
Downside: no native `choice/noul` primitives — we own the NLI-adapter
code as part of Vastra's ML surface.

**If OpenDecision wins →** shortest integration path — its API surface
matches Jev/Laya 1:1, so `src/ai/decisions.ts` becomes a thin adapter.
Downside: Python 3.13+ requirement pins the ML worker to a newer image;
project is v0.1.1 (developer preview) so we take on maturity risk.

**If nothing wins → keep Jev** and pursue Phase 1a (safety-net with Jev
+ Gemini) as originally scoped.

Raw outputs cached in `prototypes/cache/22_<candidate>_{choice,noul}_raw.json`.
Metrics CSV: `prototypes/outputs/22/22_metrics.csv`.
Gates: `prototypes/outputs/22/22_gates.csv`.
Winner: `prototypes/outputs/22/22_winner.json`.
